In [0]:
from pyspark.sql import functions as F
from datetime import datetime
from dateutil.relativedelta import relativedelta
import re

# discover all green taxi raw tables dynamically
# nyc_mobility.raw matching the green_MM_YYYY pattern, so adding a new month's
# table to raw automatically gets picked up here without editing the notebook.
all_raw_tables = spark.catalog.listTables("nyc_mobility.raw")

table_pattern = re.compile(r"^green_\d{2}_\d{4}$")
tables = [
    f"nyc_mobility.raw.{t.name}"
    for t in all_raw_tables
    if table_pattern.match(t.name)
]

assert len(tables) > 0, "FAIL: no green_MM_YYYY tables found in nyc_mobility.raw — check schema/table names"
print(f"Discovered {len(tables)} green taxi raw tables: {tables}")

# Load the union of discovered raw green taxi tables
dfs = [spark.table(t) for t in tables]
df_raw = dfs[0]
for d in dfs[1:]:
    df_raw = df_raw.unionByName(d)

df_raw = df_raw.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("lpep_dropoff_datetime") - F.unix_timestamp("lpep_pickup_datetime")) / 60
)

# derive the valid date window from the TABLE NAMES being loaded
year_months = []
for t in tables:
    table_part = t.split(".")[-1]        # e.g. "green_03_2026"
    _, month_str, year_str = table_part.split("_")
    year_months.append((int(year_str), int(month_str)))

min_year, min_month = min(year_months)
max_year, max_month = max(year_months)

window_start = datetime(min_year, min_month, 1)
window_end = datetime(max_year, max_month, 1) + relativedelta(months=1)

window_start_str = window_start.strftime("%Y-%m-%d")
window_end_str = window_end.strftime("%Y-%m-%d")

assert window_start < window_end, "FAIL: window bounds are backwards — check table name parsing"
print(f"Derived valid window from loaded tables: [{window_start_str}, {window_end_str})")

In [0]:
# STEP 1 — hard excludes: rows that are not usable trips at all
row_count_before = df_raw.count()

excluded = df_raw.filter(
    (F.col("lpep_pickup_datetime") < window_start_str) |
    (F.col("lpep_pickup_datetime") >= window_end_str) |
    (F.col("lpep_dropoff_datetime") < F.col("lpep_pickup_datetime")) |
    (F.col("trip_distance") > 100000)
)

df_kept = df_raw.exceptAll(excluded)
row_count_after = df_kept.count()

print(f"Rows before excludes: {row_count_before}")
print(f"Rows after excludes: {row_count_after}")
print(f"Rows excluded: {row_count_before - row_count_after}")

In [0]:
from pyspark.sql.types import IntegerType, DoubleType, TimestampType, StringType, DecimalType

money_cols = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
              "ehail_fee", "improvement_surcharge", "total_amount",
              "congestion_surcharge", "cbd_congestion_fee"]

int_cols = ["VendorID", "RatecodeID", "PULocationID", "DOLocationID",
            "passenger_count", "payment_type", "trip_type"]

df_typed = df_kept

for c in int_cols:
    df_typed = df_typed.withColumn(c, F.col(c).cast(IntegerType()))

for c in ["lpep_pickup_datetime", "lpep_dropoff_datetime"]:
    df_typed = df_typed.withColumn(c, F.col(c).cast(TimestampType()))

df_typed = df_typed.withColumn(
    "trip_distance", F.round(F.col("trip_distance").cast(DoubleType()), 2)
)

for c in money_cols:
    df_typed = df_typed.withColumn(c, F.round(F.col(c).cast(DoubleType()), 2))

df_typed = df_typed.withColumn("trip_duration_min", F.col("trip_duration_min").cast(DecimalType(10, 2)))
df_typed = df_typed.withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag").cast(StringType()))

In [0]:
# STEP 2 — write to silver 
target_table_silver = "nyc_mobility.clean.green_taxi"

df_typed.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table_silver)
print(f"Wrote {row_count_after} rows to {target_table_silver}")